In [1]:
from pumnum import pumnum
import pint
import unyt
from astropy import units as uastro
from numba import njit

Define a function with both @njit and @pumnum decorator
-

In [2]:
@pumnum
def function1_pumnum(a, b, c):
    aa=0
    for i in range(1000):
        aa+=(a * b / c**3 / 2)
    return aa

@njit
def function1_njit(a, b, c):
    aa=0
    for i in range(1000):
        aa+=(a * b / c**3 / 2)
    return aa

Define a, b and c as physical units with both pint and unyt
-

In [3]:
ureg = pint.UnitRegistry()

a_pint=2.5*ureg.m
b_pint=3*ureg.km
c_pint=10*ureg.s

a_unyt=2.5*unyt.m
b_unyt=3*unyt.km
c_unyt=10*unyt.s

a_astropy=2.5*uastro.m
b_astropy=3*uastro.km
c_astropy=10*uastro.s


Now, if we call function1 njit for the pint quantities, we get an error:
-

In [4]:
try:
    function1_njit(a_pint, b_pint, c_pint)
except:
    print("Error because njit is not compatible with pint")


Error because njit is not compatible with pint


But if we use the pumnum decorator, the function yields no errors, while still being numba-jitted and keeping unit consistency all along
-

In [5]:
result_pumnum_pint = function1_pumnum(a_pint, b_pint, c_pint)
print("result pumnum with pint:\n",result_pumnum_pint)

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
non-precise type pyobject
During: typing of argument at /Users/gomes/Documents/Projeto Units/venv/lib/python3.12/site-packages/pumnum.py (16)

File "../../venv/lib/python3.12/site-packages/pumnum.py", line 16:

@njit
^

During: Pass nopython_type_inference 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class 'pint.Quantity'>


For unyt, @njit does not yield an error, however it does not keep the units
-

In [ ]:
print(function1_njit(a_unyt, b_unyt, c_unyt))

With pumnum, it does:
-

In [ ]:
result_pumnum_unyt = function1_pumnum(a_unyt, b_unyt, c_unyt)
print("result pumnum with unyt:\n",result_pumnum_unyt)

Let us test now with astropy.units
-

In [ ]:
result_pumnum_astropy = function1_pumnum(a_astropy, b_astropy, c_astropy)
print(result_pumnum_astropy)

It is also possible to pass the argument "convert_to" to the pumnum decorator, and select a unit system (e.g. SI, MKS, CGS...) to convert automatically the result of the function
-
Please note that not all unit libraries have the same default unit systems. See below the list of systems for the three libraries in the notebook.

In [ ]:
@pumnum(convert_to='si')
def function1_pumnum(a, b, c):
    aa=0
    for i in range(1000):
        aa+=(a * b / c**3 / 2)
    return aa

result_pumnum_astropy_si = function1_pumnum(a_astropy, b_astropy, c_astropy)
print("SI units:", result_pumnum_astropy_si)

@pumnum(convert_to='cgs')
def function1_pumnum(a, b, c):
    aa=0
    for i in range(1000):
        aa+=(a * b / c**3 / 2)
    return aa

result_pumnum_astropy_cgs = function1_pumnum(a_astropy, b_astropy, c_astropy)
print("CGS units:", result_pumnum_astropy_cgs)

Lists of unyt systems per library
-

In [ ]:
print("pint:", dir(pint.UnitRegistry().sys))
print("unyt:", list(unyt.unit_systems.unit_system_registry))
print("astropy:", ["si", "cgs", "astrophys", "imperial", "misc", "photometric", "required_by_vounit"])

It also works for pumnum functions calling other pumnum functions
-

In [ ]:
@pumnum
def functionB(m,n):
    return m*n
    
@pumnum(convert_to="si")
def functionA(a, b, c):
    aa=a**2
    for i in range(10):
        aa+=functionB(a,c)
    return aa



In [ ]:
a_astropy=2.5*uastro.m
b_astropy=3*uastro.m
c_astropy=10*uastro.m

In [ ]:
type(a_astropy.value)

In [ ]:
print(functionA(a_astropy,b_astropy,c_astropy))